# 01 · Datos y señales: entender el punto de partida

Este cuaderno prepara Shiller y calcula momentum y reversión. Se ejecuta directamente y utiliza las funciones reutilizables de Python de `src/tfm`. No necesita un menú ni archivos de configuración.

**Al terminar sabrás qué representa cada fila, con qué meses se decide y qué significa cada señal.** No calcularemos todavía ganancias de estrategias ni entrenaremos modelos.

Selecciona el entorno `.venv` como kernel y ejecuta las celdas en orden. Para reproducir el recorrido, reinicia el kernel y ejecuta todo. El cuaderno se guarda en Git sin salidas.

## 1. Entorno y periodo del estudio

Las siguientes celdas muestran la fuente y los años del desarrollo. La historia anterior a 1927 sirve para calcular las primeras señales; el periodo desde 1990 sigue reservado para una evaluación posterior.

La siguiente celda muestra el Python elegido. Si falla `import tfm`, selecciona `.venv` o instala el proyecto desde su raíz con `python -m pip install -e ".[notebooks]"`.

In [ ]:
from tfm.data import project_root, save_result
import sys
from pathlib import Path
from math import isclose

import pandas as pd
from IPython.display import display

from tfm.data import prepare_shiller_data, calculate_signals, read_result

ROOT = project_root()
print(f"Python: {sys.executable}")
print(f"Proyecto: {ROOT}")
print("Historia previa desde: 1924-12")
print("Desarrollo: 1927-01 → 1989-12")
print("La evaluación desde 1990 está deshabilitada.")

## 2. Qué contienen los datos de Shiller

La fuente es el [archivo `ie_data.xls` de Robert J. Shiller](https://shillerdata.com/). La serie S&P Composite incluye antecedentes del S&P 500 moderno.

| Campo preparado | Significado |
|---|---|
| `period` | Mes de la observación |
| `price_monthly_mean` (P) | Promedio mensual nominal de cierres diarios, en puntos del índice |
| `dividend_annualized` (D) | Importe anualizado e interpolado de dividendos, en puntos del índice |
| `sample_role` | Historia previa (`history`) o desarrollo (`development`) |

P no representa un precio al que se pudiera operar al terminar el mes. D no es el dividendo efectivamente cobrado ese mes y no se utiliza para generar estas señales. El futuro estudio de resultados será una aproximación histórica con supuestos documentados.

## 3. Preparar los datos y las señales

**Esta celda guarda archivos en el proyecto.** Reutiliza `data/raw/shiller.xls` si existe; si falta, descarga el enlace indicado en los supuestos visibles del cuaderno. Para importar un XLS que ya tienes, sustituye `None` por `Path("/ruta/al/ie_data.xls")`.

El original de trabajo no se sustituye automáticamente. Volver a preparar regenera `shiller_monthly.parquet` y retira sus señales anteriores, la segunda llamada las vuelve a calcular. Las salidas conservan nombres fijos y los originales no se editan.

El lector selecciona los importes de la historia previa y de 1927–1989. No incorpora los importes desde 1990 ni columnas de resultados futuros.

In [ ]:
archivo_local = None  # O, por ejemplo: Path("/ruta/al/ie_data.xls")

resumen_datos = prepare_shiller_data(ROOT, file=archivo_local)
resumen_senales = calculate_signals(ROOT)

display(pd.DataFrame([
    {"resultado": "Datos mensuales", "filas": resumen_datos["rows"],
     "archivo": resumen_datos["path"]},
    {"resultado": "Señales", "filas": resumen_senales["rows"],
     "archivo": resumen_senales["path"]},
]))
print("Los controles revisan estructura y cobertura; no certifican ejecución ni exactitud financiera.")

## 4. Consultar la cobertura y las observaciones

Estas lecturas comprueban la cobertura y que las señales corresponden a los precios preparados. Hay **781 meses preparados**: 25 de historia previa (diciembre de 1924 a diciembre de 1926) y 756 de desarrollo (enero de 1927 a diciembre de 1989).

Los meses de historia previa no son meses adicionales de evaluación.

In [ ]:
mensuales, informe_mensual = read_result(ROOT, "shiller_monthly")
senales, informe_senales = read_result(ROOT, "shiller_signals")

cobertura = mensuales.groupby("sample_role", sort=False).agg(
    meses=("period", "size"), primero=("period", "min"), ultimo=("period", "max")
)
display(cobertura)
display(mensuales.head())
display(mensuales.tail())

## 5. Leer las señales

Una fila de señales distingue el **mes de decisión** (`decision_period`) del **mes objetivo posterior** (`target_period`). Por ejemplo, la decisión de diciembre de 1926 se dirige a enero de 1927.

| Señal | Significado |
|---|---|
| `+1` | Proponer una posición compradora |
| `0` | Proponer permanecer fuera del mercado |
| `−1` | Proponer una posición vendedora (solo reversión en este diseño) |
| Ausente | No existe historia suficiente; no equivale a una señal cero |

Una señal todavía no es una operación ni un resultado económico. Que usemos el mes siguiente como objetivo no convierte los promedios en precios de ejecución.

In [ ]:
columnas = ["decision_period", "target_period", "momentum_score", "reversal_score",
            "momentum_signal", "reversal_signal", "both_rules_available"]
display(senales[columnas].head(8))
print(f"Meses objetivo: {informe_senales['target_start']} → {informe_senales['target_end']}")
print(f"Meses con ambas señales disponibles: {informe_senales['both_rules_available']}")

## 6. Desglosar un ejemplo: enero de 1927

Si llamamos $m$ al mes de decisión, las reglas existentes son:

- **Momentum 12–1:** $P_{m-1}/P_{m-12}-1$. Compara dos precios separados por once variaciones mensuales y excluye el mes de decisión, el más reciente. Si es positivo propone comprar, en otro caso propone efectivo.
- **Reversión:** $P_m/P_{m-1}-1$. Si el último movimiento fue una subida propone vender, mientras que si fue una caída propone comprar. Si fue cero propone efectivo.

Para el objetivo de enero de 1927, $m$ es diciembre de 1926, es decir, momentum compara noviembre de 1926 con diciembre de 1925, reversión compara diciembre con noviembre de 1926. Momentum necesita diciembre de 1925; la historia se amplía hasta diciembre de 1924 para calcular también el filtro bajista de 24 meses, explicado en el notebook 05.

Puedes cambiar `MES_OBJETIVO` por otro mes de desarrollo para inspeccionarlo, sin ajustar las reglas.

In [ ]:
MES_OBJETIVO = "1927-01"
seleccion = senales.loc[senales["target_period"].eq(MES_OBJETIVO)]
if seleccion.empty:
    raise ValueError("Elige un mes objetivo dentro del desarrollo preparado.")
caso = seleccion.iloc[0]
m = pd.Period(caso["decision_period"], freq="M")
precios = mensuales.set_index("period")["price_monthly_mean"]

ejemplo = []
for regla, inicio, fin in [("Momentum", str(m - 12), str(m - 1)),
                           ("Reversión", str(m - 1), str(m))]:
    variacion = precios.loc[fin] / precios.loc[inicio] - 1
    campo = "momentum" if regla == "Momentum" else "reversal"
    assert isclose(variacion, float(caso[f"{campo}_score"]), abs_tol=1e-12)
    ejemplo.append({"regla": regla, "mes inicial": inicio, "P inicial": precios.loc[inicio],
                    "mes final": fin, "P final": precios.loc[fin],
                    "variación (%)": 100 * variacion, "señal": caso[f"{campo}_signal"]})

print(f"Decisión: {m} | Objetivo: {MES_OBJETIVO}")
display(pd.DataFrame(ejemplo).round(6))

La columna `variación (%)` muestra el indicador que genera cada señal. **No es la rentabilidad de aplicar esa señal al mes objetivo.** La comparación de cada fórmula con la señal guardada comprueba que hemos entendido los meses utilizados, no añade una segunda implementación al proyecto.